# AIML ZG528 — Laboratory Session
## Lab 3 + Lab 4: From Robot Behaviour to Robot Localization

### Build Your Warehouse AMR

In this laboratory session, we extend the same differential-drive warehouse
AMR developed in the previous laboratories.

The robot has:

- Wheel encoders
- IMU
- 2D LiDAR
- RGB camera
- Onboard computer

Today we add two capabilities:

**Lab 3 — Teach the robot how its motion and sensors behave**

**Lab 4 — Teach the robot where it is**

---

### Laboratory philosophy

We will follow:

**Run → Observe → Modify → Understand → Implement**

The objective is not to build a production-ready robotics system in one
laboratory session.

Instead, the objective is to understand the implementation pathway:

```text
Robot behaviour
      ↓
Data
      ↓
Machine-learning model
      ↓
Prediction
      ↓
Evaluation

Camera + Fiducial Markers
          ↓
       Features
          ↓
     ML Localization
          ↓
      Robot Pose

# Activity 0 — Environment Check

The robotics environment was configured during the environment setup
laboratory and used in Lab 1.

We will **not recreate the environment** in every laboratory.

Instead, we verify that the required packages for the current laboratory
are available in the existing Python virtual environment.

For this laboratory we will use:

- NumPy — numerical computation
- Matplotlib — visualization
- scikit-learn — machine learning

If a package is missing, do not proceed with the laboratory until the
environment has been corrected.

## If scikit-learn is not installed

The current laboratory requires **scikit-learn** for the machine-learning
experiments.

Open a terminal inside the BITS virtual machine.

First activate the existing course virtual environment:

```bash
cd ~/Lab
source .venv/bin/activate

pip install scikit-learn


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import scipy

import sklearn

print("Python version :", sys.version.split()[0])
print("NumPy version  :", np.__version__)
print("SciPy version  :", scipy.__version__)
print("scikit-learn   :", sklearn.__version__)
print("OpenCV version :", cv2.__version__)

print("\nEnvironment check completed.")

## Activity 1 — Learning How the Warehouse AMR Moves

### From a Probabilistic Model to a Data-Driven Model

Our warehouse AMR does not move exactly as commanded.

When we issue a motion command such as:

- move forward at a specified velocity
- rotate at a specified angular velocity

the actual robot motion is affected by factors such as:

- wheel slip
- actuator imperfections
- floor conditions
- payload
- mechanical differences between wheels

Therefore, the actual motion is uncertain.

In the lectures, we represented this uncertainty using a **probabilistic motion model**:

$$p(x_t \mid u_t, x_{t-1})$$

where:

- $x_{t-1}$ = previous robot state
- $u_t$ = motion command
- $x_t$ = resulting robot state

In this activity, we take the next step:

> **Can we use recorded motion data to learn the relationship between a motion command and the resulting robot motion?**

We will use a simplified dataset representing the same differential-drive warehouse AMR.

### Learning Workflow

$$
\text{Motion Commands}
\rightarrow
\text{Observed Motion}
\rightarrow
\text{Training Data}
\rightarrow
\text{ML Model}
\rightarrow
\text{Predicted Motion}
$$

We will first **inspect the data**, then train a simple machine-learning model, and finally examine whether the learned model captures the robot's motion behavior.

> **Engineering goal:** Understand the complete workflow of learning a robot motion model—not build a production-grade motion predictor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

print("Activity 1: Motion-model learning environment ready.")

### Step 1 — Inspect the Motion Data

A learning-based motion model requires examples of how the robot actually moved.

For each recorded motion command, we will consider:

- commanded linear velocity, $v$
- commanded angular velocity, $\omega$
- observed displacement in the robot's forward direction, $\Delta s$
- observed change in heading, $\Delta\theta$

The dataset represents repeated experiments with our warehouse AMR under different motion commands.

Each row is one observed motion sample:

$$
(v,\omega)
\longrightarrow
(\Delta s,\Delta\theta)
$$

Our first task is **not to train a model**.

Instead, we will:

1. load the recorded data,
2. inspect its structure,
3. look for relationships between the commanded motion and the observed motion.

> **Think first:** If the robot receives the same command multiple times, should we expect exactly the same motion every time?

The variation in the observations is what makes this a robotics modelling problem rather than a simple deterministic calculation.

In [ ]:
# Activity 1 — Create a small motion dataset

# Each row represents one observed motion of the warehouse AMR:
# [commanded linear velocity v,
#  commanded angular velocity omega,
#  observed forward displacement delta_s,
#  observed heading change delta_theta]

motion_data = np.array([
    [0.50,  0.00, 0.48,  0.01],
    [0.50,  0.00, 0.52, -0.01],
    [0.50,  0.00, 0.49,  0.02],
    [0.50,  0.00, 0.51,  0.00],

    [0.80,  0.00, 0.76,  0.02],
    [0.80,  0.00, 0.82, -0.01],
    [0.80,  0.00, 0.79,  0.01],
    [0.80,  0.00, 0.81,  0.00],

    [0.50,  0.40, 0.49,  0.38],
    [0.50,  0.40, 0.52,  0.42],
    [0.50,  0.40, 0.48,  0.39],
    [0.50,  0.40, 0.51,  0.41],

    [0.80, -0.40, 0.77, -0.38],
    [0.80, -0.40, 0.82, -0.42],
    [0.80, -0.40, 0.79, -0.39],
    [0.80, -0.40, 0.81, -0.41],
])

print("Number of motion samples:", len(motion_data))
print("\nFirst five samples:")
print(motion_data[:5])

### Step 2 — Understand the Dataset

Before applying machine learning, we should understand what each column represents.

For this activity:

| Column | Meaning |
|---|---|
| $v$ | Commanded linear velocity (m/s) |
| $\omega$ | Commanded angular velocity (rad/s) |
| $\Delta s$ | Observed forward displacement (m) |
| $\Delta\theta$ | Observed change in heading (rad) |

The first two quantities are the **inputs** to our learning problem.

The last two quantities are the **observed outcomes** of the robot's motion.

Therefore, we can think of the learning problem as:

$$
[v,\omega]
\longrightarrow
[\Delta s,\Delta\theta]
$$

The objective is to learn this relationship from examples.

> **Checkpoint:** Look at the data and identify where you can see uncertainty in the robot's motion.

In [ ]:
# Display the motion data in a readable format

print("Motion dataset:")
print("-" * 65)
print(f"{'v (m/s)':>10} {'omega (rad/s)':>15} {'delta_s (m)':>15} {'delta_theta (rad)':>20}")
print("-" * 65)

for row in motion_data:
    print(f"{row[0]:>10.2f} {row[1]:>15.2f} {row[2]:>15.2f} {row[3]:>20.2f}")

In [ ]:
# Visualize commanded velocity versus observed displacement

v = motion_data[:, 0]
delta_s = motion_data[:, 2]

plt.figure(figsize=(7, 5))
plt.scatter(v, delta_s, s=60)

plt.xlabel("Commanded linear velocity, v (m/s)")
plt.ylabel("Observed displacement, Δs (m)")
plt.title("Commanded Motion vs Observed Motion")
plt.grid(True)
plt.show()

### Step 3 — What Does the Data Tell Us?

The plot shows an important difference between a commanded motion and the motion actually observed by the robot.

For example, when the commanded velocity is:

$$
v = 0.50\ \text{m/s}
$$

the observed displacement is not exactly the same in every trial.

Similarly, for:

$$
v = 0.80\ \text{m/s}
$$

the observed displacement varies slightly.

This means that the robot's motion cannot be described perfectly by a single deterministic relationship.

A learning-based model can instead use the observed examples to approximate this relationship.

> **Checkpoint:** Why might a learned model be useful when the same motion command produces slightly different outcomes?

In [ ]:
# Prepare inputs and outputs for the learning model

X = motion_data[:, :2]   # [v, omega]
y = motion_data[:, 2:]   # [delta_s, delta_theta]

print("Input shape:", X.shape)
print("Output shape:", y.shape)

print("\nFirst five input samples [v, omega]:")
print(X[:5])

print("\nFirst five output samples [delta_s, delta_theta]:")
print(y[:5])
print("\nEach row represents one motion sample.")

### Step 4 — Training Data and Testing Data

A machine-learning model should not be evaluated only on the data it was trained on.

If we train a model using all available observations and then test it using the same observations, we are asking:

> **Can the model reproduce data that it has already seen?**

That does not tell us whether the model can make useful predictions for new motion commands.

Instead, we divide the available data into two parts:

- **Training data** — used to learn the relationship between motion commands and observed motion.
- **Testing data** — kept aside and used to evaluate the learned model on observations it did not use during training.

Conceptually:

$$
\text{Recorded Data}
\rightarrow
\begin{cases}
\text{Training Data} & \rightarrow \text{Learn the model} \\
\text{Testing Data} & \rightarrow \text{Evaluate the model}
\end{cases}
$$

For this small demonstration, we will use:

- **75%** of the samples for training
- **25%** of the samples for testing

> **Engineering idea:** A useful learned motion model should capture the underlying relationship, not simply memorize the observations used during training.

In [ ]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining input shape:", X_train.shape)
print("Testing input shape:", X_test.shape)

print("\nTraining output shape:", y_train.shape)
print("Testing output shape:", y_test.shape)

In [ ]:
# Train a simple machine-learning model

motion_model = LinearRegression()

motion_model.fit(X_train, y_train)

print("Motion model training complete.")
print("Number of input features:", X_train.shape[1])
print("Number of predicted outputs:", y_train.shape[1])

In [ ]:
# Inspect the parameters learned by the linear model

print("Model coefficients:")
print(motion_model.coef_)

print("\nModel intercept:")
print(motion_model.intercept_)

In [ ]:
# Compare observed motion with the motion predicted by the learned model

y_pred_train = motion_model.predict(X_train)

plt.figure(figsize=(7, 5))

plt.scatter(
    y_train[:, 0],
    y_pred_train[:, 0],
    s=60
)

# Ideal prediction line: predicted = observed
min_val = min(y_train[:, 0].min(), y_pred_train[:, 0].min())
max_val = max(y_train[:, 0].max(), y_pred_train[:, 0].max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Observed displacement, Δs (m)")
plt.ylabel("Predicted displacement, Δs (m)")
plt.title("Observed vs Predicted Robot Motion")
plt.grid(True)
plt.show()

In [ ]:
# Use the trained model to predict motion for unseen test samples

y_pred_test = motion_model.predict(X_test)

print("Test inputs [v, omega]:")
print(X_test)

print("\nActual motion [delta_s, delta_theta]:")
print(y_test)

print("\nPredicted motion [delta_s, delta_theta]:")
print(y_pred_test)

In [ ]:
# Calculate the prediction error on the unseen test data

errors = y_test - y_pred_test

print("Prediction errors [delta_s, delta_theta]:")
print(errors)

print("\nMean absolute error:")
print(np.mean(np.abs(errors), axis=0))

In [ ]:
# Compare actual and predicted displacement on unseen test samples

plt.figure(figsize=(7, 5))

plt.scatter(
    range(len(y_test)),
    y_test[:, 0],
    s=70,
    label="Actual"
)

plt.scatter(
    range(len(y_pred_test)),
    y_pred_test[:, 0],
    s=70,
    marker="x",
    label="Predicted"
)

plt.xlabel("Test sample")
plt.ylabel("Displacement, Δs (m)")
plt.title("Actual vs Predicted Motion on Unseen Data")
plt.xticks(range(len(y_test)))
plt.legend()
plt.grid(True)
plt.show()

### Engineering Checkpoint — Learned Motion Model

We have now followed a complete data-driven modelling workflow:

$$
\text{Motion Commands}
\rightarrow
\text{Observed Motion}
\rightarrow
\text{Training Data}
\rightarrow
\text{Learned Model}
\rightarrow
\text{Predicted Motion}
\rightarrow
\text{Evaluation}
$$

The learned model can approximate the observed motion of the warehouse AMR.

However, remember:

- the dataset used here is small and simplified,
- the model is only an approximation,
- prediction accuracy depends on the quality and diversity of the training data,
- a model trained in one operating condition may not perform equally well in another.

> **Think:** What might happen if the warehouse floor, payload, or operating speed changes significantly?

This is one of the central challenges of learning-based robotics:

> **A model can perform well on the data it represents, but may fail when the operating conditions change.**

## Activity 2 — Learning a LiDAR Sensor Model

### From a Probabilistic Measurement Model to a Data-Driven Model

Our warehouse AMR uses a **2D LiDAR** to observe its surroundings.

In the lectures, we represented the LiDAR measurement using a probabilistic measurement model:

$$
p(z_t \mid x_t,m)
$$

where:

- $x_t$ = robot state
- $m$ = map
- $z_t$ = sensor measurement

The model captures the fact that a sensor does not always return exactly the ideal measurement.

For example, the same physical distance may produce slightly different LiDAR readings because of:

- sensor noise,
- surface properties,
- incidence angle,
- reflections,
- measurement limitations.

In this activity, we take the next step:

> **Can we use recorded sensor data to learn how the LiDAR behaves?**

We will use a simplified dataset representing measurements from the LiDAR mounted on our warehouse AMR.

### Learning Workflow

$$
\text{Robot/Environment State}
\rightarrow
\text{Sensor Observation}
\rightarrow
\text{Training Data}
\rightarrow
\text{ML Model}
\rightarrow
\text{Predicted Measurement}
$$

We will first **inspect the sensor data**, then train a simple machine-learning model, and finally examine how well the learned model represents the observed sensor behaviour.

> **Engineering goal:** Understand the workflow for learning a sensor model—not build a production-grade LiDAR model.

### Step 1 — Inspect the Sensor Data

A learned sensor model requires examples of how the LiDAR behaves under different conditions.

For each recorded observation, we will consider:

- the **true distance** to an object,
- the **beam angle** relative to the object,
- the **measured distance** reported by the LiDAR.

Each row represents one sensor observation:

$$
(\text{true distance},\text{beam angle})
\longrightarrow
\text{measured distance}
$$

Our first task is **not to train a model**.

Instead, we will:

1. create and inspect the recorded sensor observations,
2. examine how the measured distance differs from the true distance,
3. identify patterns that a learning model might capture.

> **Think first:** If the LiDAR observes the same physical distance multiple times, should we expect exactly the same measurement every time?

The variation in the observations is the starting point for modelling **sensor uncertainty**.

In [ ]:
# Activity 2 — Create a LiDAR sensor dataset
# The measurements include a small angle-dependent effect.

# Each row represents:
# [true distance (m), beam angle (degrees), measured distance (m)]

sensor_data = np.array([
    [2.0, -30, 2.10],
    [2.0, -15, 2.05],
    [2.0,   0, 2.01],
    [2.0,  15, 2.05],
    [2.0,  30, 2.11],

    [3.0, -30, 3.11],
    [3.0, -15, 3.06],
    [3.0,   0, 3.01],
    [3.0,  15, 3.05],
    [3.0,  30, 3.12],

    [4.0, -30, 4.13],
    [4.0, -15, 4.07],
    [4.0,   0, 4.02],
    [4.0,  15, 4.06],
    [4.0,  30, 4.14],

    [5.0, -30, 5.15],
    [5.0, -15, 5.08],
    [5.0,   0, 5.03],
    [5.0,  15, 5.07],
    [5.0,  30, 5.16],
])

print("Number of sensor observations:", len(sensor_data))

print("\nFirst five observations:")
print(sensor_data[:5])

### Step 2 — Understand the Sensor Dataset

Before training a model, let's make the relationship explicit.

For this activity:

| Column | Meaning |
|---|---|
| True distance | Actual distance to the observed object (m) |
| Beam angle | LiDAR beam angle relative to the object (degrees) |
| Measured distance | Distance reported by the LiDAR (m) |

The first two quantities describe the **physical sensing situation**.

The last quantity is the **sensor observation**.

Therefore, our learning problem is:

$$
[\text{true distance},\text{beam angle}]
\longrightarrow
\text{measured distance}
$$

Notice that the measured distance is not exactly equal to the true distance.

For example:

$$
\text{True distance}=2.0\text{ m}
$$

may produce:

$$
\text{Measured distance}=2.08\text{ m}
$$

in one observation, and a different value in another observation.

> **Checkpoint:** Look at the dataset and identify at least two observations where the measured distance differs from the true distance.

This difference is what a sensor model needs to represent.

In [ ]:
# Visualize how the LiDAR measurement changes with beam angle

true_distance = sensor_data[:, 0]
beam_angle = sensor_data[:, 1]
measured_distance = sensor_data[:, 2]

plt.figure(figsize=(7, 5))

plt.scatter(
    beam_angle,
    measured_distance - true_distance,
    s=60
)

plt.axhline(0, linestyle="--")

plt.xlabel("Beam angle (degrees)")
plt.ylabel("Measurement error (m)")
plt.title("LiDAR Measurement Error vs Beam Angle")
plt.grid(True)
plt.show()

### Step 3 — Define the Learning Problem

We can now formulate the sensor-learning problem in the same way as we did for the motion model.

The inputs to the model are:

$$
X =
[\text{true distance},\text{beam angle}]
$$

The target is:

$$
y =
\text{measured distance}
$$

Therefore:

$$
X \longrightarrow y
$$

or, more explicitly:

$$
[\text{true distance},\text{beam angle}]
\longrightarrow
\text{measured distance}
$$

This is a **supervised learning** problem because each input example is paired with a known target measurement.

### Classical vs Learning-Based View

**Classical sensor model**

$$
p(z\mid x,m)
$$

describes the probability of observing a measurement given the robot state and map.

**Learning-based sensor model**

$$
\hat{z}=f_\theta(d,\alpha)
$$

learns an approximation of the sensor response from recorded examples.

where:

- $d$ = true distance
- $\alpha$ = beam angle
- $\hat{z}$ = predicted sensor measurement
- $f_\theta$ = learned model

> **Key idea:** We are not replacing the concept of a sensor model. We are changing how the model is obtained—from explicitly specifying it to learning an approximation from data.

In [ ]:
# Prepare inputs and target for the sensor-learning problem

X_sensor = sensor_data[:, :2]   # [true distance, beam angle]
y_sensor = sensor_data[:, 2]    # [measured distance]

print("Input shape:", X_sensor.shape)
print("Target shape:", y_sensor.shape)

print("\nFirst five input samples [true distance, beam angle]:")
print(X_sensor[:5])

print("\nFirst five target values [measured distance]:")
print(y_sensor[:5])

### Step 4 — Split the Data into Training and Testing Sets

As with the motion model, we should not evaluate the learned sensor model only on the observations used for training.

We will divide the sensor observations into:

- **Training data** — used to learn the relationship between distance, angle, and measurement.
- **Testing data** — kept aside to evaluate the model on observations it did not use during training.

Conceptually:

$$
\text{Sensor Data}
\rightarrow
\begin{cases}
\text{Training Data} & \rightarrow \text{Learn the sensor model} \\
\text{Testing Data} & \rightarrow \text{Evaluate the model}
\end{cases}
$$

For this demonstration, we will use:

- **75%** of the observations for training
- **25%** of the observations for testing

> **Engineering idea:** A learned sensor model should capture the relationship between the physical sensing conditions and the measurement, rather than simply memorize individual observations.

In [ ]:
# Split the sensor data into training and testing sets

X_train_sensor, X_test_sensor, y_train_sensor, y_test_sensor = train_test_split(
    X_sensor,
    y_sensor,
    test_size=0.25,
    random_state=42
)

print("Training samples:", len(X_train_sensor))
print("Testing samples:", len(X_test_sensor))

print("\nTraining input shape:", X_train_sensor.shape)
print("Testing input shape:", X_test_sensor.shape)

print("\nTraining target shape:", y_train_sensor.shape)
print("Testing target shape:", y_test_sensor.shape)

In [ ]:
# Train a simple machine-learning model for the LiDAR sensor

sensor_model = LinearRegression()

sensor_model.fit(X_train_sensor, y_train_sensor)

print("Sensor model training complete.")
print("Number of input features:", X_train_sensor.shape[1])
print("Number of predicted outputs:", 1)

What does “learn” mean here?
The model tries to find a relationship of the form:
$$ \hat z = b + w_1d + w_2\alpha $$
where:
\(d\) = true distance
\(\alpha\) = beam angle
\(w_1,w_2\) = learned coefficients
\(b\) = learned intercept
\(\hat z\) = predicted LiDAR measurement
Students do not need to calculate these coefficients manually. LinearRegression estimates them from the training data.
Important distinction from Activity 1
In Activity 1:
$$ [v,\omega]\rightarrow[\Delta s,\Delta\theta] $$
so there were two predicted outputs.
Here:
$$ [d,\alpha]\rightarrow z $$
so there is one predicted output.

In [ ]:
# Inspect the parameters learned by the sensor model

print("Model coefficients:")
print(sensor_model.coef_)

print("\nModel intercept:")
print(sensor_model.intercept_)

In [ ]:
# Predict LiDAR measurements for the unseen test observations

y_pred_sensor = sensor_model.predict(X_test_sensor)

print("Test inputs [true distance, beam angle]:")
print(X_test_sensor)

print("\nActual LiDAR measurements:")
print(y_test_sensor)

print("\nPredicted LiDAR measurements:")
print(y_pred_sensor)

In [ ]:
# Calculate the prediction error on the unseen sensor data

sensor_errors = y_test_sensor - y_pred_sensor

print("Prediction errors (actual - predicted):")
print(sensor_errors)

print("\nMean absolute error:")
print(np.mean(np.abs(sensor_errors)))

### Step 5 — Visualize Actual vs Predicted LiDAR Measurements

A numerical error metric tells us **how much** the model is wrong.

A visualization can help us understand **where and how** it is wrong.

We will compare:

- actual LiDAR measurements from the test data,
- predictions made by the learned sensor model.

If the learned model captures the relationship reasonably well, the predicted values should remain close to the actual measurements.

> **Think:** Are the prediction errors approximately the same for all beam angles, or does the model appear to perform differently at different angles?

This question is important because a model may have a good average error while still performing poorly under particular sensing conditions.

In [ ]:
# Compare actual and predicted LiDAR measurements on unseen test data

plt.figure(figsize=(7, 5))

plt.scatter(
    y_test_sensor,
    y_pred_sensor,
    s=70
)

# Ideal prediction line: predicted = actual
min_val = min(y_test_sensor.min(), y_pred_sensor.min())
max_val = max(y_test_sensor.max(), y_pred_sensor.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Actual LiDAR measurement (m)")
plt.ylabel("Predicted LiDAR measurement (m)")
plt.title("Actual vs Predicted LiDAR Measurements")
plt.grid(True)
plt.show()

### Engineering Checkpoint — Learned Sensor Model

We have now followed a complete data-driven sensor-modelling workflow:

$$
\text{Sensor Observations}
\rightarrow
\text{Training Data}
\rightarrow
\text{Learned Model}
\rightarrow
\text{Predicted Measurements}
\rightarrow
\text{Evaluation}
$$

The learned model can approximate the observed behaviour of the warehouse AMR's LiDAR.

However, remember:

- the dataset used here is small and simplified,
- the model is only an approximation,
- prediction accuracy depends on the quality and diversity of the training data,
- a model trained under one sensing condition may not perform equally well under another.

### Think

Suppose the warehouse AMR is moved from:

- a clean warehouse floor,
- to a warehouse containing highly reflective metal surfaces.

**Would you expect the learned sensor model to behave exactly the same? Why or why not?**

> **Key engineering idea:** A learned sensor model is only as reliable as the data and operating conditions it represents.

We have now seen two examples of data-driven robotics modelling:

1. **Motion model:** commands → observed robot motion
2. **Sensor model:** physical situation → observed sensor measurement

In the next activity, we will use machine learning for a different task:

> **Estimating the robot's pose from visual observations of fiducial markers.**

# Activity 3 — Machine Learning for Robot Localization

## From Visual Observations to Robot Pose

Our warehouse AMR is equipped with an RGB camera.

In a structured warehouse, the robot may encounter **fiducial markers** placed at known locations.

A camera observation of a marker can provide information about the robot's position.

In the lecture, we considered the idea:

$$
\text{Camera Observation}
\rightarrow
\text{Robot Pose}
$$

In this activity, we will explore how machine learning can be used to learn this relationship from recorded examples.

### Learning Workflow

$$
\text{Camera}
\rightarrow
\text{Fiducial Marker Detection}
\rightarrow
\text{Features}
\rightarrow
\text{Training Data}
\rightarrow
\text{ML Model}
\rightarrow
\text{Estimated Robot Pose}
$$

We will use a simplified dataset representing observations made by the camera of our warehouse AMR.

The objective is not to build a complete camera-localization system.

Instead, we will understand the **implementation pathway**:

1. represent visual observations as numerical features,
2. associate those observations with known robot poses,
3. train a supervised learning model,
4. use the trained model to estimate the pose of a new observation.

> **Engineering goal:** Understand how machine learning can be used as one component of a robot localization pipeline.

In [ ]:
# Activity 3 — Machine-learning localization

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

print("Activity 3: ML localization environment ready.")

In [ ]:
# Activity 3 — Create a simplified fiducial-marker localization dataset

# Each row represents one observation of a known fiducial marker
# from the camera mounted on the warehouse AMR.
#
# Features:
#   marker_distance    -> estimated distance from camera to marker (m)
#   marker_bearing     -> estimated horizontal bearing of marker (degrees)
#   marker_orientation -> estimated relative orientation of marker (degrees)
#
# Targets:
#   robot_x            -> robot position in the warehouse (m)
#   robot_y            -> robot position in the warehouse (m)
#   robot_theta        -> robot heading (degrees)

# Generate a larger, geometrically consistent fiducial-marker dataset

rng = np.random.default_rng(42)

# Known fiducial marker position in the warehouse world frame
marker_x = 4.0
marker_y = 3.0

localization_rows = []

# Robot positions across the warehouse
for robot_x in np.arange(1.0, 8.0, 1.0):
    for robot_y in np.arange(1.0, 6.0, 1.0):

        # Vector from robot to the fiducial marker
        dx = marker_x - robot_x
        dy = marker_y - robot_y

        # Ideal marker observation
        distance = np.hypot(dx, dy)
        bearing = np.degrees(np.arctan2(dy, dx))

        # Generate repeated observations with small measurement noise
        for _ in range(3):

            measured_distance = distance + rng.normal(0, 0.03)
            measured_bearing = bearing + rng.normal(0, 0.7)

            localization_rows.append([
                measured_distance,
                measured_bearing,
                0.0,          # marker orientation
                robot_x,
                robot_y,
                0.0           # robot orientation
            ])

localization_data = np.array(localization_rows)

print("Dataset shape:", localization_data.shape)
print("Number of observations:", len(localization_data))

print("\nColumns:")
print("[marker_distance, marker_bearing, marker_orientation, robot_x, robot_y, robot_theta]")

print("\nFirst five observations:")
print(localization_data[:5])

### Understanding the Localization Dataset

Each row represents one observation of the fiducial marker from the warehouse AMR.

| Feature / Target | Meaning |
|---|---|
| `marker_distance` | Distance from robot to the fiducial marker (m) |
| `marker_bearing` | Bearing of the marker relative to the robot (degrees) |
| `marker_orientation` | Orientation of the detected marker |
| `robot_x` | Ground-truth robot x-position (m) |
| `robot_y` | Ground-truth robot y-position (m) |
| `robot_theta` | Ground-truth robot orientation (degrees) |

The first three quantities are the **observations** available to the localization model.

The last three quantities describe the **ground-truth robot pose**:

$$
\mathbf{x} =
\begin{bmatrix}
x & y & \theta
\end{bmatrix}^{T}
$$

For this first ML localization exercise, we will predict only:

$$
\mathbf{y} =
\begin{bmatrix}
x & y
\end{bmatrix}^{T}
$$

We are not yet learning robot orientation because all robot orientations in this simplified dataset are fixed at:

$$
\theta = 0^\circ
$$

A machine-learning model cannot learn a meaningful relationship for a quantity that does not vary in the training data.

### How the Dataset Was Created

We assume a fiducial marker is fixed at a known location in the warehouse:

$$
M=(4,3)
$$

We generate observations for many different robot positions.

For each robot position, the ideal marker distance and bearing are calculated from the relative geometry. Small measurement noise is then added to represent realistic sensor uncertainty.

Three observations are generated for each robot position.

Therefore:

$$
7\times5=35\text{ robot positions}
$$

and

$$
35\times3=105\text{ observations}
$$

### Learning Problem

The ML model learns the relationship:

$$
[\text{distance},\text{bearing},\text{orientation}]
\rightarrow
[\text{robot }x,\text{robot }y]
$$

During **training**:

$$
\text{Marker Observation}
+
\text{Known Robot Pose}
\rightarrow
\text{ML Model}
$$

During **inference**:

$$
\text{New Marker Observation}
\rightarrow
\text{ML Model}
\rightarrow
\text{Estimated Robot Position}
$$

> **Important:** This exercise demonstrates the ML localization workflow. It is a simplified simulation of the sensing and learning process, not a complete camera-based localization system.

In [ ]:
# Separate the marker observations from the ground-truth robot pose

X_localization = localization_data[:, :3]   # [distance, bearing, orientation]
y_localization = localization_data[:, 3:5]  # [robot_x, robot_y]

print("Input shape:", X_localization.shape)
print("Target shape:", y_localization.shape)

print("\nFirst five input samples [distance, bearing, orientation]:")
print(X_localization[:5])

print("\nFirst five target values [robot_x, robot_y]:")
print(y_localization[:5])

In [ ]:
# Visualize the marker observations used for localization

marker_distance = localization_data[:, 0]
marker_bearing = localization_data[:, 1]

plt.figure(figsize=(8, 5))

plt.scatter(
    marker_bearing,
    marker_distance,
    alpha=0.7
)

plt.xlabel("Marker Bearing (degrees)")
plt.ylabel("Marker Distance (m)")
plt.title("Fiducial Marker Observations")
plt.grid(True)
plt.show()

### Formulating the ML Localization Problem

We now have a set of marker observations and corresponding ground-truth robot positions.

The ML problem is:

$$
\mathbf{X} =
\begin{bmatrix}
d & \beta & \phi
\end{bmatrix}
\rightarrow
\mathbf{y} =
\begin{bmatrix}
x & y
\end{bmatrix}
$$

where:

- $d$ = marker distance
- $\beta$ = marker bearing
- $\phi$ = marker orientation
- $x,y$ = robot position

This is a **supervised regression problem** because the training data contains known input-output pairs.

### Classical Localization

A conventional fiducial-marker localization pipeline would be:

$$
\text{Camera Image}
\rightarrow
\text{Marker Detection}
\rightarrow
\text{Geometric Pose Estimation}
\rightarrow
(x,y,\theta)
$$

The geometry of the camera, marker, and robot is explicitly used to calculate the pose.

### Learning-Based Localization

In a learning-based approach, we use labelled observations to learn the relationship:

$$
[d,\beta,\phi]
\rightarrow
\text{ML Model}
\rightarrow
(\hat{x},\hat{y})
$$

The model does not explicitly solve the geometric equations. Instead, it learns a mapping from examples.

### Training vs Inference

**During training:**

$$
\text{Marker Observation}
+
\text{Known Robot Position}
\rightarrow
\text{ML Model}
$$

**During inference:**

$$
\text{New Marker Observation}
\rightarrow
\text{ML Model}
\rightarrow
\text{Estimated Robot Position}
$$

> **Key idea:** Machine learning changes how the relationship is obtained. It does not remove the need for meaningful sensor observations.

In a real warehouse AMR, the marker observations would ultimately come from the RGB camera through a fiducial-marker detection and pose-estimation pipeline.

For this laboratory, those observations are provided as pre-generated data so that we can focus on the machine-learning component.

In [ ]:
# Split the localization data into training and testing sets

X_train_loc, X_test_loc, y_train_loc, y_test_loc = train_test_split(
    X_localization,
    y_localization,
    test_size=0.25,
    random_state=42
)

print("Training samples:", len(X_train_loc))
print("Testing samples:", len(X_test_loc))

print("\nTraining input shape:", X_train_loc.shape)
print("Testing input shape:", X_test_loc.shape)

print("\nTraining target shape:", y_train_loc.shape)
print("Testing target shape:", y_test_loc.shape)

In [ ]:
# Train a Random Forest model to estimate robot position

localization_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

localization_model.fit(
    X_train_loc,
    y_train_loc
)

print("Localization model training complete.")
print("Training samples:", len(X_train_loc))
print("Input features:", X_train_loc.shape[1])
print("Predicted outputs:", y_train_loc.shape[1])

In [ ]:
# Predict robot positions for the unseen test observations

y_pred_loc = localization_model.predict(X_test_loc)

print("Actual robot positions:")
print(y_test_loc)

print("\nPredicted robot positions:")
print(y_pred_loc)

In [ ]:
# Visualize actual and predicted robot positions

plt.figure(figsize=(7, 5))

plt.scatter(
    y_test_loc[:, 0],
    y_test_loc[:, 1],
    s=80,
    label="Actual Position"
)

plt.scatter(
    y_pred_loc[:, 0],
    y_pred_loc[:, 1],
    s=80,
    marker="x",
    label="Predicted Position"
)

plt.xlabel("Robot X Position (m)")
plt.ylabel("Robot Y Position (m)")
plt.title("ML-Based Robot Localization")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Calculate the localization prediction error

position_error = np.linalg.norm(
    y_test_loc - y_pred_loc,
    axis=1
)

mae_x = np.mean(np.abs(y_test_loc[:, 0] - y_pred_loc[:, 0]))
mae_y = np.mean(np.abs(y_test_loc[:, 1] - y_pred_loc[:, 1]))
mean_position_error = np.mean(position_error)

print("Mean Absolute Error in X:", round(mae_x, 3), "m")
print("Mean Absolute Error in Y:", round(mae_y, 3), "m")
print("Mean Euclidean Position Error:", round(mean_position_error, 3), "m")

## Engineering Checkpoint — ML-Based Localization

We have now implemented a simplified learning-based localization pipeline.

### What We Did

$$
\text{Marker Observations}
\rightarrow
\text{Training Data}
\rightarrow
\text{Random Forest}
\rightarrow
\text{Predicted Robot Position}
$$

We:

1. Generated marker observations for multiple robot positions.
2. Added small measurement noise to represent sensor uncertainty.
3. Separated observations into inputs and ground-truth targets.
4. Split the data into training and test sets.
5. Trained a Random Forest regression model.
6. Predicted robot positions for unseen observations.
7. Visualized actual and predicted positions.
8. Quantified the localization error.

### Think Like a Robotics Engineer

**Question 1**

If the localization error is too large, should we immediately choose a more complex ML model?

**Question 2**

What could be changed in the training data before changing the model?

**Question 3**

Why might a model perform well in one part of the warehouse but poorly in another?

**Question 4**

What would need to change before deploying this approach on a real warehouse AMR?

### Key Takeaway

> **A machine-learning localization system is only as useful as the observations, training data, model, and evaluation process that support it.**

In a real system, the complete pipeline would extend further:

$$
\text{RGB Camera}
\rightarrow
\text{Fiducial Detection}
\rightarrow
\text{Marker Features}
\rightarrow
\text{ML Model}
\rightarrow
\text{Robot Pose}
$$

Our laboratory has focused on the **ML component** of this pipeline.